# RAG-over-PDF study tool — v1

This notebook is the "main" that wires the four pipeline stages together:

`parse → chunk → index → query`

Each stage lives in the `rag` package and writes a file to disk, so you can
inspect every intermediate artifact. v1 has **no OCR** and **no LLM** — the
output is a `context.txt` you paste into any chat model yourself.

**First time:** `pip install -r requirements.txt`. The embedding model
(~130 MB) downloads on first use and is cached afterwards.


## Setup
Put your slide PDFs in the `pdfs/` folder next to this notebook.

In [ ]:
from pathlib import Path
import rag

# --- paths -------------------------------------------------------------
PDF_DIR   = Path("D:\Ca' Foscari\Semester 2\Information rtrival and web search")            # drop your .pdf slide decks here
WORK_DIR  = Path("artifacts")       # where intermediate files are written
WORK_DIR.mkdir(exist_ok=True)

RECORDS   = WORK_DIR / "records.jsonl"
CHUNKS    = WORK_DIR / "chunks.jsonl"
INDEX     = WORK_DIR / "index"          # -> index.npy + index.chunks.json
CONTEXT   = WORK_DIR / "context.txt"

# --- retrieval knobs ---------------------------------------------------
TOP_K  = 5     # how many independent chunks seed the result
WINDOW = 1     # how many neighbor slides (x-1, x+1) each seed drags in


## Stage 1 — Parse
PDFs → one record per slide (text + merged notes), saved as JSONL.

In [ ]:
import json

records = rag.parse_directory(PDF_DIR)
rag.save_records(records, RECORDS)

n_docs  = len({r["doc_id"] for r in records})
n_empty = sum(1 for r in records if not r["has_text"])
print(f"Parsed {len(records)} slides across {n_docs} document(s).")
print(f"{n_empty} slide(s) had no extractable text (OCR candidates for later).")
print("\nExample record:")
print(json.dumps(records[0], indent=2, ensure_ascii=False)[:500])


## Stage 2 — Chunk
v1 rule: one slide = one chunk. Empty (image-only) slides are dropped for now.

In [ ]:
chunks = rag.chunk_records(records, drop_empty=True)
rag.save_records(chunks, CHUNKS)   # same JSONL writer works for chunks
print(f"{len(chunks)} chunks ready to index.")
print("Example chunk id:", chunks[0]["chunk_id"])


## Stage 3 — Embed & index
Embed every chunk with a local model and persist the matrix + chunks.

In [ ]:
model = rag.load_model()                 # downloads on first run, then cached
index = rag.build_index(chunks, model)
rag.save_index(index, INDEX)
print(f"Indexed {len(index)} chunks with model '{index.model_name}'.")
print("Embedding matrix shape:", index.embeddings.shape)


## Stage 4 — Query
Type your question below. The query is embedded, the top-k chunks retrieved,
their slide-neighbors pulled in, and the assembled context written to
`context.txt` (and printed). That file is your prompt — paste it into any chat
model along with your question.

In [ ]:
question = "Explain how the chunking step relates to neighbor retrieval."

query_vec = rag.embed_query(question, model)
hits      = rag.retrieve(query_vec, index, k=TOP_K)

print("Top-k seeds:")
for chunk, score in hits:
    print(f"  {score:.3f}  {chunk['doc_id']} slide {chunk['slide_num']}")

expanded = rag.expand_neighbors(hits, index, window=WINDOW)
context  = rag.assemble_context(question, expanded)

CONTEXT.write_text(context, encoding="utf-8")
print(f"\nWrote {CONTEXT}  ({len(expanded)} chunks after neighbor expansion)\n")
print(context)


### One-call shortcut
Once the index exists you can skip the manual steps. `answer_to_file` runs
embed → retrieve → expand → write in one go.

In [ ]:
rag.answer_to_file(
    "What metadata does each chunk carry, and why?",
    index, model, CONTEXT, k=TOP_K, window=WINDOW,
)
print(CONTEXT.read_text(encoding="utf-8"))


### Reusing a saved index
In a fresh session you don't need to re-parse. Load the index and the model,
then query:

```python
import rag
model = rag.load_model()
index = rag.load_index("artifacts/index")
rag.answer_to_file("your question", index, model, "artifacts/context.txt")
```
